In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\Raw_data_1Day_2024_site_1426_Narela_Delhi_DPCC_1Day.csv")

In [3]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),...,MP-Xylene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg),VWS (m/s)
0,2024-01-01,171.91,278.42,5.20,19.19,24.40,44.98,3.04,1.43,12.07,...,NaN,8.47,75.52,0.22,137.29,0.00,0.00,22.20,981.06,NaN
1,2024-01-02,174.65,288.56,6.59,20.19,26.77,40.21,6.82,1.53,14.87,...,NaN,8.31,71.60,0.21,104.07,0.00,0.00,42.48,980.17,NaN
2,2024-01-03,213.22,318.41,14.47,22.33,36.78,46.07,2.89,1.80,14.68,...,NaN,7.64,80.56,0.22,187.80,0.00,0.00,22.98,979.83,NaN
3,2024-01-04,181.68,293.01,11.10,20.04,24.14,39.13,2.33,1.61,6.48,...,NaN,7.04,84.28,0.29,90.28,0.00,0.00,14.83,980.12,NaN
4,2024-01-05,135.38,237.00,8.53,22.79,18.57,39.39,3.35,1.74,20.88,...,NaN,8.49,84.80,0.30,120.81,0.00,0.00,13.78,978.18,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,135.79,202.33,21.65,31.89,34.55,41.93,2.46,0.94,3.28,...,NaN,15.42,90.21,0.94,12.74,0.80,0.80,47.11,980.84,NaN
362,2024-12-28,80.63,142.45,24.01,29.55,35.24,39.52,7.10,1.05,4.60,...,NaN,15.78,94.77,0.35,13.15,0.09,0.09,55.61,982.68,NaN
363,2024-12-29,73.58,108.50,17.13,23.43,26.37,38.25,1.83,0.94,3.84,...,NaN,15.11,90.51,0.58,13.57,0.00,0.00,81.59,981.24,NaN
364,2024-12-30,81.71,124.25,15.98,21.49,24.41,36.21,2.96,0.94,3.28,...,NaN,14.01,85.35,0.49,13.42,0.00,0.00,77.33,980.76,NaN


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (366, 21)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['Xylene (µg/m³)']
Dropped rows (>70% NaN): 1
Missing values after imputation:
 Timestamp          0
PM2.5 (µg/m³)      0
PM10 (µg/m³)       0
NO (µg/m³)         0
NO2 (µg/m³)        0
NOx (ppb)          0
NH3 (µg/m³)        0
SO2 (µg/m³)        0
CO (mg/m³)         0
Ozone (µg/m³)      0
Benzene (µg/m³)    0
Toluene (µg/m³)    0
AT (°C)            0
RH (%)             0
WS (m/s)           0
WD (deg)           0
RF (mm)            0
TOT-RF (mm)        0
SR (W/mt2)         0
BP (mmHg)          0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (365, 20)
    Timestamp  PM2.5 (µg/m³)  PM10 (µg/m³)  NO (µg/m³)  NO2 (µg/m³)  \
0  2024-01-01         171.91        278.42        5.20        19.19   
1  2024-01-02         174.65        288.56        6.59        20.19   
2  2024-01-03         213.22        318.41       14.47        22.33   
3  2024-01-04         181.68        293.01       11.10        20.04   
4  2024-01-05         135.38        237.00        8.53        22.79   

   NOx (ppb)  NH3 (µg/m³)  SO2 (µg/m³)  CO (mg/m³)  Ozone (µg/m³)  \
0      24.40        44.98         3.04        1.43          12.07   
1      26.77        40.21         6.82        1.53          14.87   
2      36.78        46.07         2.89        1.80          14.68   
3      24.14        39.13         2.33        1.61           6.48   
4      18.57        39.39         3.35        1.74          20.88   

   Benzene (µg/m³)  Toluene (µg/m³)  AT (°C)  RH (%)  WS (m/s)  WD (deg)  \
0             1.32            84.24     8.47   75.52      0

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),Benzene (µg/m³),Toluene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg)
0,2024-01-01,1.238859,0.485032,-0.974386,-0.937723,-0.208390,1.864126,-1.058390,1.115234,-1.340744,-1.257730,-1.246237,-2.089763,0.960206,-1.201125,-0.102616,0.0,0.0,-1.910603,-0.706980
1,2024-01-02,1.282952,0.573825,-0.769643,-0.855446,-0.000593,1.231260,-0.541699,1.433026,-1.250238,-1.250768,-1.594665,-2.108904,0.718343,-1.229673,-0.615444,0.0,0.0,-1.249677,-0.988730
2,2024-01-03,1.903644,0.835211,0.391054,-0.679374,0.877061,2.008743,-1.078894,2.291065,-1.256380,-1.021000,-1.106324,-2.189058,1.271172,-1.201125,0.677124,0.0,0.0,-1.885183,-1.096364
3,2024-01-04,1.396083,0.612792,-0.105335,-0.867788,-0.231186,1.087970,-1.155441,1.687260,-1.521432,-0.638055,-1.626564,-2.260838,1.500695,-1.001290,-0.828325,0.0,0.0,-2.150792,-1.004558
4,2024-01-05,0.650996,0.122331,-0.483888,-0.641527,-0.719551,1.122466,-1.016016,2.100390,-1.055974,-0.328217,-1.445145,-2.087370,1.532778,-0.972742,-0.357023,0.0,0.0,-2.185011,-1.618709
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
360,2024-12-27,0.657594,-0.181263,1.448645,0.107192,0.681539,1.459463,-1.137671,-0.441949,-1.624868,1.214009,2.420077,-1.258315,1.866574,0.854326,-2.025337,0.0,0.0,-1.098786,-0.776626
361,2024-12-28,-0.230073,-0.705613,1.796265,-0.085336,0.742037,1.139714,-0.503425,-0.092377,-1.582201,0.778843,1.017744,-1.215247,2.147924,-0.830002,-2.019007,0.0,0.0,-0.821770,-0.194133
362,2024-12-29,-0.343526,-1.002902,0.782864,-0.588870,-0.035664,0.971215,-1.223786,-0.441949,-1.606766,-0.390881,0.419419,-1.295401,1.885083,-0.173399,-2.012524,0.0,0.0,0.024918,-0.649997
363,2024-12-30,-0.212693,-0.864984,0.613472,-0.748486,-0.207513,0.700556,-1.069325,-0.441949,-1.624868,-0.592798,-0.080992,-1.426997,1.566713,-0.430331,-2.014839,0.0,0.0,-0.113915,-0.801952


In [10]:
df.to_excel('narela2024.xlsx', index=False)